# Replace the contents of `e-qsms-reproducibility` on GitHub

This notebook puts the final reproducibility package for the AQT paper into
`github.com/sudhirsingh-wq/e-qsms-reproducibility`:

* **Replaces** everything in the repository (the old `README.md` and
  `EQSMS_publish_to_github.ipynb`) with the final package: `protocol/`,
  `verification/`, `notebooks/`, `data/`, `figures/`, `README.md` and `requirements.txt`.
* **Updates the description** to the paper's title, and adds topics.
* **Checks** the result: it lists the files now on GitHub and runs the quick
  verification scripts from a fresh clone.

The package is **embedded in this notebook**, so there is nothing to upload. The push is
an ordinary commit, not a force-push: the old files are removed by the commit, and the history is kept.

**You need** a GitHub personal access token that can write to the repository:
* **Classic token** (`github.com/settings/tokens` → *Tokens (classic)*): tick **repo**.
* **Fine-grained token** (`github.com/settings/personal-access-tokens`): give it access to
  `e-qsms-reproducibility` with **Contents: Read and write** and **Administration: Read and write**
  (the description change needs Administration).

Run the cells in order with **Runtime → Run all**; the whole notebook takes about two minutes.

## 1. Token and author

In [ ]:
import getpass

OWNER = "sudhirsingh-wq"
REPO  = "e-qsms-reproducibility"
DESCRIPTION = ("Code, IBM hardware data and verification suite for 'Publicly Verifiable "
               "Quantum Secure Multiparty Summation with Entanglement-Sourced Commitments' "
               "(Advanced Quantum Technologies, submitted)")
TOPICS = ["quantum-cryptography", "secure-multiparty-computation", "quantum-computing",
          "ghz-state", "pedersen-commitment", "qiskit", "reproducibility"]

TOKEN = None
try:                                    # Colab secrets panel (key icon), if you stored it there
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
    print("token read from Colab secrets")
except Exception:
    pass
if not TOKEN:
    TOKEN = getpass.getpass("GitHub token (hidden): ").strip()

AUTHOR_NAME  = input("Commit author name [Sudhir Singh]: ").strip() or "Sudhir Singh"
AUTHOR_EMAIL = input("Commit author e-mail [sudhirsingh@igdtuw.ac.in]: ").strip() or "sudhirsingh@igdtuw.ac.in"
assert TOKEN, "a token is required"
print(f"target: {OWNER}/{REPO}   token length: {len(TOKEN)}")

## 2. Check the token

In [ ]:
import json, urllib.request, urllib.error

def api(path, payload=None, method="GET", accept="application/vnd.github+json"):
    req = urllib.request.Request(
        "https://api.github.com" + path,
        data=json.dumps(payload).encode() if payload is not None else None,
        method=method,
        headers={"Authorization": f"Bearer {TOKEN}", "Accept": accept,
                 "X-GitHub-Api-Version": "2022-11-28", "Content-Type": "application/json"})
    with urllib.request.urlopen(req) as r:
        body = r.read()
        return json.loads(body) if body else {}

try:
    print("authenticated as:", api("/user")["login"])
except urllib.error.HTTPError as e:
    raise SystemExit(f"token rejected ({e.code}): expired or mistyped")

info = api(f"/repos/{OWNER}/{REPO}")
perms = info.get("permissions", {})
print("repository   :", info["html_url"], "(private)" if info["private"] else "(public)")
print("can push     :", perms.get("push"))
print("can admin    :", perms.get("admin"))
assert perms.get("push"), "this token cannot push to the repository"

print("\ncurrently on GitHub:")
for item in api(f"/repos/{OWNER}/{REPO}/contents"):
    print("  ", item["path"])

## 3. Unpack the final package

The package (30 files, about 136 KB) is stored in the next cell as base64 text.

In [ ]:
PAYLOAD = (
    "..."
    "AAAeAB4AtAoAAEGQAAAAAA=="
)

import base64, io, zipfile, pathlib, shutil

NEW = pathlib.Path("/content/eqsms_new")
if NEW.exists():
    shutil.rmtree(NEW)
with zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(PAYLOAD)))) as z:
    z.extractall(NEW)
NEW = NEW / "e-qsms-reproducibility"

files_new = sorted(p.relative_to(NEW).as_posix() for p in NEW.rglob("*") if p.is_file())
for f in files_new:
    print("  ", f)
print(f"\n{len(files_new)} files")

### Credential scan

An extra check, before anything leaves Colab, that no token or key is in the files.

In [ ]:
import re
PATTERNS = {
    "IBM Cloud CRN":     r"crn:v1:bluemix:[a-z]+:[a-z]+:[a-z]",
    "GitHub token":      r"gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}",
    "AWS key":           r"AKIA[0-9A-Z]{16}",
    "private key block": r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
    "IBM API key":       r"\b[A-Za-z0-9_-]{44}\b(?=[\"'])",
}
hits = [(f, lab) for f in files_new for lab, pat in PATTERNS.items()
        if re.search(pat, (NEW / f).read_text(errors="ignore"))]
if hits:
    for f, lab in hits:
        print("   ", f, "->", lab)
    raise SystemExit("possible credential found: stop and check before pushing")
print("credential scan clean")

## 4. Replace the repository contents and push

The token goes to git as a one-off HTTP header, so it is not written into `.git/config`,
the remote URL or the history.

In [ ]:
import subprocess, os

CLONE = pathlib.Path("/content/eqsms_repo")
if CLONE.exists():
    shutil.rmtree(CLONE)

AUTH = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
URL  = f"https://github.com/{OWNER}/{REPO}.git"

def git(*args, cwd=CLONE, secret=False):
    cmd = ["git"] + (["-c", f"http.extraheader={AUTH}"] if secret else []) + list(args)
    r = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        print((r.stdout + r.stderr).replace(TOKEN, "***"))
        raise SystemExit("git " + args[0] + " failed")
    return r.stdout.strip()

git("clone", "-q", URL, str(CLONE), cwd="/content", secret=True)
branch = git("rev-parse", "--abbrev-ref", "HEAD")
print("cloned, branch:", branch)

# remove everything except .git, then copy the final package in
for p in CLONE.iterdir():
    if p.name == ".git":
        continue
    shutil.rmtree(p) if p.is_dir() else p.unlink()
for f in files_new:
    dst = CLONE / f
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(NEW / f, dst)

git("config", "user.name", AUTHOR_NAME)
git("config", "user.email", AUTHOR_EMAIL)
git("add", "-A")
print("\nchanges to be committed:")
print(git("status", "--short"))

if git("status", "--porcelain"):
    git("commit", "-q", "-m",
        "Final reproducibility package for the Advanced Quantum Technologies submission\n\n"
        "Protocol implementation, verification suite, IBM hardware data (ibm_marrakesh,\n"
        "job dbd62os24mrc73ckqkn0), derived results and figure scripts.")
    git("push", "-q", "origin", branch, secret=True)
    print("\npushed:", git("log", "-1", "--format=%h %s"))
else:
    print("\nnothing to change: the repository already holds this package")

## 5. Update the description and topics

In [ ]:
try:
    api(f"/repos/{OWNER}/{REPO}", {"description": DESCRIPTION}, "PATCH")
    api(f"/repos/{OWNER}/{REPO}/topics", {"names": TOPICS}, "PUT")
    print("description and topics updated")
except urllib.error.HTTPError as e:
    print(f"could not update the description ({e.code}).")
    print("Your token lacks Administration access. Set it by hand instead: open the")
    print(f"repository page, click the gear next to 'About', and paste:\n\n{DESCRIPTION}")

## 6. Check what is now on GitHub

In [ ]:
info = api(f"/repos/{OWNER}/{REPO}")
print("description:", info["description"])
print("visibility :", "private" if info["private"] else "public")
tree = api(f"/repos/{OWNER}/{REPO}/git/trees/{info['default_branch']}?recursive=1")
on_github = sorted(t["path"] for t in tree["tree"] if t["type"] == "blob")
missing = set(files_new) - set(on_github)
extra   = set(on_github) - set(files_new)
print(f"files on GitHub: {len(on_github)}   missing: {sorted(missing) or 'none'}   left over: {sorted(extra) or 'none'}")
assert not missing and not extra, "the repository does not match the package"
if info["private"]:
    print("\nThe repository is PRIVATE: reviewers cannot open it. Make it public under Settings.")
print("\n", info["html_url"])

## 7. Optional: run the verification from a fresh clone

This does what a reviewer would do. The quick scripts take under a minute in total and
must reproduce the stored results exactly. `verify_brief.py` (about 2 minutes) and
`sessions.py` (about 6 minutes) are left out here; run them the same way if you want them.

In [ ]:
import filecmp, sys
CHECK = pathlib.Path("/content/eqsms_check")
if CHECK.exists():
    shutil.rmtree(CHECK)
subprocess.run(["git", "clone", "-q", f"https://github.com/{OWNER}/{REPO}.git", str(CHECK)],
               check=True)  # public clone, no token
ref = pathlib.Path("/content/eqsms_ref"); shutil.rmtree(ref, ignore_errors=True)
shutil.copytree(CHECK / "data" / "derived", ref)

for s in ["quantum_layer", "toy_privacy", "exposure", "confirm_preparer_issues"]:
    r = subprocess.run([sys.executable, str(CHECK / "verification" / f"{s}.py")],
                       capture_output=True, text=True)
    print(f"{s:26s}", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-800:])

for f in ["quantum_layer.json", "toy_privacy.json", "exposure.json"]:
    same = filecmp.cmp(CHECK / "data/derived" / f, ref / f, shallow=False)
    print(f"{f:26s}", "reproduced exactly" if same else "DIFFERS")

r = subprocess.run([sys.executable, str(CHECK / "figures" / "make_figs_v3.py")], capture_output=True, text=True)
print("figures                   ", "rebuilt" if r.returncode == 0 else "FAILED\n" + r.stderr[-800:])

## Afterwards

* **Delete the token** if you made it only for this: `github.com/settings/tokens`.
* **Optional: archive a release.** Tag a `v1.0` release on GitHub and connect the
  repository to Zenodo for a DOI; if you get one, it can go into the Data Availability Statement.

| symptom | cause |
|---|---|
| `token rejected (401)` | token expired or mistyped |
| `this token cannot push` | classic token without **repo**, or fine-grained without **Contents: write** |
| description not updated (403) | fine-grained token without **Administration: write**; set it by hand as printed |
| `git push failed` | someone pushed in the meantime: re-run from Step 4 |